## In this Project we will implement complete Medallion Architecture, We will import files from ADLS and then dump into Bronze layer. We are taking SAMPLE CUSTOMER DATA For this.  

In [ ]:
from snowflake.snowpark.context import get_active_session

from snowflake.snowpark.functions import (
        col, trim, upper, initcap, lit, to_date, when, is_null, 
        round as sf_round, sum as sf_sum, count as sf_count, 
        year, month, dayofmonth, dayofweek, dayname, monthname,
        quarter, date_part, iff, coalesce, length
)

from snowflake.snowpark.types import (
    DecimalType, 
    StringType, 
    IntegerType, 
    FloatType, 
    BooleanType, 
    DateType, 
    TimestampType,
    ArrayType,
    MapType,
    VariantType
)

session = get_active_session()

In [ ]:
# Target Databses / Schema for this Demo
session.sql("CREATE DATABASE IF NOT EXISTS YOUTUBEDEMO").collect()
session.sql("CREATE OR REPLACE SCHEMA YOUTUBEDEMO.BRONZE").collect()
session.sql("CREATE OR REPLACE SCHEMA YOUTUBEDEMO.SILVER").collect()
session.sql("CREATE OR REPLACE SCHEMA YOUTUBEDEMO.GOLD").collect()

src = "SNOWFLAKE_SAMPLE_DATA.TPCH_SF1"

In [ ]:
session.sql("""
    CREATE OR REPLACE FILE FORMAT YOUTUBEDEMO.BRONZE.CSV_FF
        TYPE = CSV
        FIELD_DELIMITER = ','
        SKIP_HEADER = 1
        FIELD_OPTIONALLY_ENCLOSED_BY = '"'
        NULL_IF = ('', 'NULL')
    """).collect()

In [ ]:
session.sql("""
    CREATE TABLE IF NOT EXISTS YOUTUBEDEMO.BRONZE.CUSTOMER_EXT (
        CUSTKEY NUMBER,
        NAME    STRING,
        ADDRESS STRING,
        NATIONKEY NUMBER,
        PHONE   STRING,
        ACCTBAL NUMBER(12,2),
        MKTSEGMENT STRING,
        COMMENT STRING
    )
    """).collect()

In [ ]:
# Storage integration --> Same like SQL query
session.sql("""
    CREATE OR REPLACE STORAGE INTEGRATION S3_TPCH_INT
        TYPE = EXTERNAL_STAGE
        STORAGE_PROVIDER = S3
        ENABLED = TRUE
        STORAGE_AWS_ROLE_ARN = 'aen:aws:iam::<account_id>:role/<snowflake-access-role>'
        STORAGE_ALLOWD_LOCATIONS = ('s3://cly-tpch-raw/customer/')
    """).collect()

# Create External Stage after authenticate in AWS IAM
session.sql("""
    CREATE OR REPLACE STAGE CLY_TPCH_DEMO.BRONZE.S3_CUSTOMER_STAGE
        STORAGE_INTEGRATION = S3_TPCH_INT
        URL = 's3://cly-tpch-raw/customer/'
        FILE_FORMAT = CLY_TPCH_DEMO.BRONZE.CSV_FF
    """).collect()

# COPY INTO COMMAND
session.sql("""
    COPY INTO CLY_TPCH_DEMO.BRONZE.CUSTOMER_EXT
    FROM @CLY_TPCH_DEMO.BRONZE.S3_CUSTOMER_STAGE
    PATTERN = '.*customer.*[.]csv'
    ON_ERROR = 'CONTINUE'
    """).collect()

In [ ]:
df_customer_raw = session.table(f"{src}.CUSTOMER")
df_orders_raw = session.table(f"{src}.ORDERS")
df_lineitem_raw = session.table(f"{src}.LINEITEM")
df_nation_raw = session.table(f"{src}.NATION")
df_region_raw = session.table(f"{src}.REGION")

print('Row Counts (Bronze):')
print('Customer :', df_customer_raw.count())
print('Orders :', df_orders_raw.count())
print('Lineitem :', df_lineitem_raw.count())
print('Nation :', df_nation_raw.count())
print('Region :', df_region_raw.count())

In [ ]:
df_customer_raw.write.save_as_table("YOUTUBEDEMO.BRONZE.CUSTOMER", mode="overwrite")
df_orders_raw.write.save_as_table("YOUTUBEDEMO.BRONZE.ORDERS", mode="overwrite")
df_lineitem_raw.write.save_as_table("YOUTUBEDEMO.BRONZE.LINEITEM", mode="overwrite")
df_nation_raw.write.save_as_table("YOUTUBEDEMO.BRONZE.NATION", mode="overwrite")
df_region_raw.write.save_as_table("YOUTUBEDEMO.BRONZE.REGION", mode="overwrite")

## NOW LET's CLEAN THE DATA ---> SILVER LAYER

In [ ]:
df_customer_silver = (
    df_customer_raw\
    .select(
    col("C_CUSTKEY").alias("CUSTOMER_KEY"),
    trim(col("C_NAME")).alias("CUSTOMER_NAME"),
    trim(col("C_ADDRESS")).alias("CUSTOMER_ADDRESS"),
    col("C_NATIONKEY").alias("NATION_KEY"),
    trim(col("C_PHONE")).alias("CUSTOMER_PHONE"),
    upper(trim(col("C_MKTSEGMENT"))).alias("MARKET_SEGMENT")
    )\
    .filter(col("CUSTOMER_KEY").is_not_null())
    .dropDuplicates(["CUSTOMER_KEY"])
    .withColumn(
        "MARKET_SEGMENT",
        coalesce(col("MARKET_SEGMENT"), lit("UNKNOWN"))
        )
)

df_customer_silver.write.save_as_table("YOUTUBEDEMO.SILVER.CUSTOMER_SILVER", mode = "overwrite")

In [ ]:
# Silver Nation Table
df_nation_silver = (
        df_nation_raw\
        .select(
            col("N_NATIONKEY").alias("NATION_KEY"),
            initcap(trim(col("N_NAME"))).alias("NATION_NAME"),
            col("N_REGIONKEY").alias("REGION_KEY")
        )\
        .dropDuplicates(["NATION_KEY"])
    )
    
df_nation_silver.write.save_as_table("YOUTUBEDEMO.SILVER.NATION_SILVER", mode = "overwrite")

# Sliver Region Table
df_region_silver = (
        df_region_raw\
        .select(       
            col("R_REGIONKEY").alias("REGION_KEY"),
            initcap(trim(col("R_NAME"))).alias("REGION_NAME"),
        )\
        .dropDuplicates(["REGION_KEY"])
    )

df_region_silver.write.save_as_table("YOUTUBEDEMO.SILVER.REGION_SILVER", mode = "overwrite")

In [ ]:
# SILVER ORDER TABLE
df_orders_silver = (
df_orders_raw \
    .select(
        col("O_ORDERKEY").alias("ORDER_KEY"),
        col("O_CUSTKEY").alias("CUSTOMER_KEY"),
        upper(trim(col("O_ORDERSTATUS"))).alias("ORDER_STATUS"),
        col("O_TOTALPRICE").cast(DecimalType(15,2)).alias("TOTAL_PRICE"),
        to_date(col("O_ORDERDATE")).alias("ORDER_DATE"),
        initcap(trim(col("O_ORDERPRIORITY"))).alias("ORDER_PRIORITY"),
        trim(col("O_CLERK")).alias("CLERK")
    ) \
    .filter(col("ORDER_KEY").is_not_null()) \
    .dropDuplicates(["ORDER_KEY"]) \
    .withColumn(
        "ORDER_STATUS",
        when(col("ORDER_STATUS") == "O", lit("OPEN"))
        .when(col("ORDER_STATUS") == "F", lit("FULFILLED"))
        .when(col("ORDER_STATUS") == "P", lit("PENDING"))
        .otherwise(col("ORDER_STATUS"))
    ) \
)

df_orders_silver.write.save_as_table("YOUTUBEDEMO.SILVER.ORDERS_SILVER", mode = "overwrite")

In [ ]:
# SILVER LINEITEM TABLE
df_lineitem_silver = (
df_lineitem_raw \
    .select(
        col("L_ORDERKEY").alias("ORDER_KEY"),
        col("L_LINENUMBER").alias("LINE_NUMBER"),
        col("L_PARTKEY").alias("PART_KEY"),  # Fixed typo: PARTY_KEY -> PART_KEY
        col("L_SUPPKEY").alias("SUPPLIER_KEY"),
        col("L_QUANTITY").cast(DecimalType(12,2)).alias("QUANTITY"),
        col("L_EXTENDEDPRICE").cast(DecimalType(15,2)).alias("EXTENDED_PRICE"),
        col("L_DISCOUNT").cast(DecimalType(5,4)).alias("DISCOUNT"),
        col("L_TAX").cast(DecimalType(5,4)).alias("TAX"),
        upper(trim(col("L_RETURNFLAG"))).alias("RETURN_FLAG"),
        upper(trim(col("L_LINESTATUS"))).alias("LINE_STATUS"),
        to_date(col("L_SHIPDATE")).alias("SHIP_DATE"),
        trim(col("L_SHIPMODE")).alias("SHIP_MODE")  # Fixed column overwrite: SHIP_DATE -> SHIP_MODE
    ) \
    .filter(col("ORDER_KEY").is_not_null() & col("LINE_NUMBER").is_not_null()) \
    .dropDuplicates(["ORDER_KEY", "LINE_NUMBER"]) \
    .withColumn( 
        "NET_REVENUE",
        sf_round(col("EXTENDED_PRICE") * (lit(1) - col("DISCOUNT")), 2)
    ) \
)

df_lineitem_silver.write.save_as_table("YOUTUBEDEMO.SILVER.LINEITEM_SILVER", mode = "overwrite")

In [ ]:
print('Row Counts (Silver):')
print('Customer :', df_customer_silver.count())
print('Orders :', df_orders_silver.count())
print('Lineitem :', df_lineitem_silver.count())
print('Nation :', df_nation_silver.count())
print('Region :', df_region_silver.count())

In [ ]:
df_orders_silver.dtypes

In [ ]:
df_orders_silver.describe()

In [ ]:
# JOIN Customer with nation and region --> Select Customer_key, Customer_name ---- from CUSTOMER LEFT JOIN ON C.NATION_KEY = N.NATION_KEY

df_customer_enriched = (
    df_customer_silver
    .join(df_nation_silver, on="NATION_KEY", how="left")
    .join(df_region_silver, on="REGION_KEY", how="left")
    .select(
        "CUSTOMER_KEY", "CUSTOMER_NAME", "CUSTOMER_ADDRESS", "CUSTOMER_PHONE",
        "MARKET_SEGMENT", "NATION_NAME", "REGION_NAME"
    )
)

df_customer_enriched

In [ ]:
# Orders  + Lineitem --> transaction grain, then bring in customer attributes
df_order_lineitem = df_orders_silver.join(
    df_lineitem_silver, on="ORDER_KEY", how="inner"
)

In [ ]:
df_fact_base = (
    df_order_lineitem
    .join(df_customer_enriched, on="CUSTOMER_KEY", how="left")
    .select(
        "ORDER_KEY", "LINE_NUMBER", "CUSTOMER_KEY", "ORDER_DATE", "SHIP_DATE",
        "ORDER_STATUS", "ORDER_PRIORITY", "QUANTITY", "EXTENDED_PRICE",
        "DISCOUNT", "TAX", "NET_REVENUE", "RETURN_FLAG", "LINE_STATUS",
        "SHIP_MODE", "MARKET_SEGMENT", "NATION_NAME", "REGION_NAME"
    )
)

print("Fact grain row count (order X lineitem):", df_fact_base.count())

In [ ]:
df_fact_base.show(4)

In [ ]:
CREATE OR REPLACE TABLE YOUTUBEDEMO.SILVER.DIM_DATE_STAGE AS
WITH date_range AS
    (
        SELECT FULL_DATE FROM (
            SELECT
                DATEADD(DAY, SEQ4(), (SELECT '1990-01-01' as MINDATE)) AS FULL_DATE
            FROM TABLE(GENERATOR(ROWCOUNT => 20000))
            )
        WHERE FULL_DATE <= (SELECT '2000-01-01' AS MAX_DATE)
    )
SELECT 
    TO_NUMBER(TO_CHAR(FULL_DATE, 'YYYYMMDD'))   AS DATE_KEY,
    FULL_DATE                                   AS FULL_DATE,
    DAY(FULL_DATE)                              AS DAY_OF_MONTH,
    MONTH(FULL_DATE)                            AS MONTH_NUMBER,
    MONTHNAME(FULL_DATE)                        AS MONTH_NAME,
    QUARTER(FULL_DATE)                          AS QUARTER_NUMBER,
    YEAR(FULL_DATE)                             AS YEAR_NUMBER,
    DAYOFWEEK(FULL_DATE)                        AS DAY_OF_WEEK_NUMBER,
    DAYNAME(FULL_DATE)                          AS DAY_NAME,
    IFF(DAYOFWEEK(FULL_DATE) IN (0,6), TRUE, FALSE) AS IS_WEEKEND
FROM date_range

In [ ]:
df_dim_date = session.table("YOUTUBEDEMO.SILVER.DIM_DATE_STAGE")

print("DIM_DATE row count:", df_dim_date.count())

## NOW BUILD GOLD LAYER - BUILD DIM_CUSTOMER, DIM_DATE, and FACT_ORDERS WITH SURROGATE JOIN

In [ ]:
df_dim_customer_gold = df_customer_enriched ## already at customer grain

df_fact_orders_gold = (
    df_fact_base
    .withColumn("ORDER_DATE_KEY", date_part("year", col("ORDER_DATE")) * 10000
                + date_part("month", col("ORDER_DATE")) * 100
                + date_part("day", col("ORDER_DATE")))
    .withColumn("SHIP_DATE_KEY", date_part("year", col("SHIP_DATE")) * 10000
                + date_part("month", col("SHIP_DATE")) * 100
                + date_part("day", col("SHIP_DATE")))
    .select(
        "ORDER_KEY", "LINE_NUMBER", "CUSTOMER_KEY", "ORDER_DATE_KEY", "SHIP_DATE_KEY",
        "ORDER_STATUS", "ORDER_PRIORITY", "QUANTITY", "EXTENDED_PRICE", "DISCOUNT",
        "TAX", "NET_REVENUE", "RETURN_FLAG", "LINE_STATUS", "SHIP_MODE", "MARKET_SEGMENT",
        "NATION_NAME", "REGION_NAME"        
    )
)

## NOW SAVE THIS OUTPUT TABLES INTO GOLD LAYER

In [ ]:
# DIM_DUSTOMER TABLE
df_dim_customer_gold.write.save_as_table("YOUTUBEDEMO.GOLD.DIM_CUSTOMER", mode="overwrite")

# DIM_DATE TABLE
df_dim_date.write.save_as_table("YOUTUBEDEMO.GOLD.DIM_DATE", mode="overwrite")

# FACT_ORDERS TABLE
df_fact_orders_gold.write.save_as_table("YOUTUBEDEMO.GOLD.FACT_ORDERS", mode="overwrite")

In [ ]:
print("Gold Layer Written: DIM_CUSTOMER, DIM_DATE, and FACT_ORDERS")
print("DIM_CUSTOMERS ROWS :", df_dim_customer_gold.count())
print("DIM_DATE ROWS :", df_dim_date.count())
print("FACT_ORDERS ROWS :", df_fact_orders_gold.count())